In [ ]:
# Integrantes:
# Iair
# Julieta Medina Fernández
# Gonzalo

Librerias y Dependencias

In [ ]:
# Instalamos paquetes necesarios para el trabajo práctico
# Hugging Face
!pip install datasets
!pip install datasets huggingface_hub

# Para etapa de fine tuning
# Para evitar problemas con TrainingArguments al ejecutar el tuneo en BETO
!pip uninstall -y transformers
!rm -rf /root/.cache/huggingface/transformers
!rm -rf /usr/local/lib/python3.11/dist-packages/transformers*
!pip install transformers==4.52.3
!pip install pysentimiento

In [ ]:
# Para Lematización
!pip install spacy nltk
!python -m spacy download es_core_news_sm

In [ ]:
!pip install pysentimiento

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import os

# Utilizados para la carga de los datasets de hugging face, descomentar en caso de corresponder y ejecutar la instalación de paquetes necesarios
from datasets import load_dataset, Dataset, DatasetDict
from huggingface_hub import login

# Para obtener datasets privados de hugging face
login()

In [ ]:
# Stopwords y Tokens
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
# Utilizado para normalizacion de los textos de los corpus
import re

# Definimos un patrón regex para detectar emojis y ciertos caracteres especiales unicode.
emoji_pattern = re.compile("["
         u"\U0001F600-\U0001F64F"  # emojis de caritas felices, tristes, etc.
         u"\U0001F300-\U0001F5FF"  # símbolos y pictogramas
         u"\U0001F680-\U0001F6FF"  # transporte y mapas
         u"\U0001F1E0-\U0001F1FF"  # banderas
         u"\U00002500-\U00002BEF"  # símbolos chinos
         u"\U00002702-\U000027B0"  # Símbolos varios y pictogramas
         u"\U000024C2-\U0001F251"  # Símbolos encerrados y pictogramas variados
         u"\U0001f926-\U0001f937"  # Emojis gestuales y expresivos
         u"\U00010000-\U0010ffff"  # Todo el resto de caracteres Unicode complementarios
         u"\u200d"                 # Zero Width Joiner (ZWJ): carácter invisible que une emojis o caracteres para formar combinaciones de emojis
         u"\u2640-\u2642"          # Símbolos de género
         u"\u2600-\u2B55"          # símbolos comunes extendidos
         u"\u23cf"                 # Botón de expulsar
         u"\u23e9"                 # Botón de avance rápido
         u"\u231a"                 # Relojes
         u"\ufe0f"                 # marcas de variación
         u"\u3030"                 #  Símbolo de onda
                           "]+", flags=re.UNICODE)

In [ ]:
# Para preprocesamiento de los corpus
import nltk
from nltk.corpus import stopwords # Para stopwords
from nltk.stem import SnowballStemmer # Para Stemming
import spacy # Para Lematizacion

# Para baseline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer # Para vectorización de baseline

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix

# Para probar modelos especificos
from pysentimiento import create_analyzer

# Para fine tuning
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, get_scheduler, EarlyStoppingCallback
from transformers.trainer_utils import get_last_checkpoint
from torch.optim import AdamW
import torch

La siguiente sección corresponde al confeccionado de los datasets que vamos a estar utilizando, a partir de los encontrados en internet. Comprende la primera parte del preprocesamiento de los datos. Los datasets resultantes fueron exportados en formato .csv para realizar el preprocesamiento formal de los tweets sobre los corpus que formamos para los propósitos buscados. El siguiente código es el utilizado para esta primera parte, pero no es necesario ejecutarlo: lo mantenemos como documentación inicial de la conformación de los corpus iniciales con los que vamos a trabajar.

In [ ]:
# Preprocesamiento de los datasets encontrados
# Para nuestro análisis fusionamos los tweets de 4 datasets distintos en un único dataset a partir del cual vamos a realizar todo el trabajo

In [ ]:
# Convertimos datasets de hugging face a dataframes de pandas

def convert_dataset_to_dataframe(dataset_name):
    # Importamos el dataset: viene separado en dos distintos, el de entrenamiento y el de testeo
      ds = load_dataset(dataset_name)
      # print(ds)

      # Convertimos cada dataset a un dataframe, obtenemos una lista con ambos dataframes resultantes
      dfs = [ds[split].to_pandas() for split in ds.keys()]
      # Los concatenamos
      df = pd.concat(dfs, ignore_index=True)

      # Imprimimos las dimensiones del dataframe resultante de la unión entre ambos ambos y los primeros 10 ejemplos que posee
      print(df.shape)
      print(df.head(10))
      return df

In [ ]:
# Dataset 1: TwitterSentimentDataset (GitHub)
# Link de referencia: https://github.com/garnachod/TwitterSentimentDataset
# Tres archivos txt con los textos de tweets en español, podemos inferir
#   text: the annotated post
#   labels: 1 positive, 0 negative, 3 neutral

def create_dataframe_from_txt(file_path):
    # Leemos el archivo de texto
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    # Creamos el dataframe
    df = pd.DataFrame([line.strip() for line in lines], columns=['text'])
    return df

# Por cada archivo de texto creamos un dataframe
dfPos = create_dataframe_from_txt('tweets_pos_clean.txt')
dfNeg = create_dataframe_from_txt('tweets_neg_clean.txt')
dfNeu = create_dataframe_from_txt('tweets_clean.txt')

# Vemos los primeros 5 tweets de cada conjunto
#print(dfPos.head(5))
#print(dfNeg.head(5))
#print(dfNeu.head(5))

# Categorizamos los tweets según su polaridad de sentimiento
# Obs: 2 queda reservado para cuando no puede inferirse sentimiento
dfPos['label'] = 1
dfNeg['label'] = 0
dfNeu['label'] = 3

# Unimos los 3 datasets en uno único
dfTwitterSentimentDataset = pd.concat([dfPos, dfNeg, dfNeu], ignore_index=True)
# Mezclamos aleatoriamente los registros
dfTwitterSentimentDataset = dfTwitterSentimentDataset.sample(frac=1, random_state=27).reset_index(drop=True)

# Guardamos el dataset resultante
dfTwitterSentimentDataset.to_csv('TwitterSentimentDataset.csv', index=False, encoding='utf-8')

# Imprimimos las dimensiones del dataset resultante y los últimos 10 ejemplos que posee
print(dfTwitterSentimentDataset.shape)
print(dfTwitterSentimentDataset.tail(10))

In [ ]:
# Dataset 2: TASS-2019 (Hugging Face)
# Link de referencia: https://huggingface.co/datasets/mrm8488/tass-2019
# Columns:
#   sentence: the annotated post
#   sentiments: P o Positivo, N o Negativo, NEU o Neutro, y NONE o sin sentimiento detectable
#   labels: 1 positive, 0 negative, 2 (o -1) no detectable feeling, 3 neutral

df_tass_2019 = convert_dataset_to_dataframe("mrm8488/tass-2019")

# Unificamos en el corpus de SentimentAnalysis las categorias de "no emoción detectable"
df_tass_2019['labels'] = df_tass_2019['labels'].replace(-1, 2)

print('Cantidades de ejemplos por cada clase de sentimiento:')
print(df_tass_2019['labels'].value_counts())

df_tass_2019.to_csv('tass_2019.csv', index=False, encoding='utf-8')

In [ ]:
# Dataset 2: TASS-2019 (Hugging Face)
# Link de referencia: https://huggingface.co/datasets/mrm8488/tass-2019
# Columns:
#   sentence: the annotated post
#   sentiments: P o Positivo, N o Negativo, NEU o Neutro, y NONE o sin sentimiento detectable
#   labels: 1 positive, 0 negative, 2 (o -1) no detectable feeling, 3 neutral

df_tass_2019 = convert_dataset_to_dataframe("mrm8488/tass-2019")

# Unificamos en el corpus de SentimentAnalysis las categorias de "no emoción detectable"
df_tass_2019['labels'] = df_tass_2019['labels'].replace(-1, 2)

print('Cantidades de ejemplos por cada clase de sentimiento:')
print(df_tass_2019['labels'].value_counts())

df_tass_2019.to_csv('tass_2019.csv', index=False, encoding='utf-8')

In [ ]:
# Creación del dataset para detección de sentimiento polarizado
# Fusionamos dfTwitterSentimentDataset con df_tass_2019

# Revisamos las columnas de df_tass_2019
print(df_tass_2019.columns)

# Eliminamos la columna sentiments dado que no vamos a necesitarla
df_tass_2019 = df_tass_2019.drop(columns=['sentiments'])

# Renombramos las columnas del segundo dataset para que coincidan con las del primero
df_tass_2019 = df_tass_2019.rename(columns={'sentence': 'text', 'labels': 'label'})

# Unimos dfTwitterSentimentDataset con df_tass_2019
dfSentimentAnalysis = pd.concat([dfTwitterSentimentDataset, df_tass_2019], ignore_index=True)
# Mezclamos aleatoriamente los registros
dfSentimentAnalysis = dfSentimentAnalysis.sample(frac=1, random_state=12).reset_index(drop=True)

# Guardamos el dataset resultante para detección de sentimiento polarizado
dfSentimentAnalysis.to_csv('SentimentAnalysis_1.csv', index=False, encoding='utf-8')

# Imprimimos las dimensiones del dataset resultante y los últimos 10 ejemplos que posee
print(dfSentimentAnalysis.shape)
print(dfSentimentAnalysis.tail(10))

In [ ]:
# Creación del dataset para detección de sentimiento polarizado
# Fusionamos dfTwitterSentimentDataset con df_tass_2019

# Revisamos las columnas de df_tass_2019
print(df_tass_2019.columns)

# Eliminamos la columna sentiments dado que no vamos a necesitarla
df_tass_2019 = df_tass_2019.drop(columns=['sentiments'])

# Renombramos las columnas del segundo dataset para que coincidan con las del primero
df_tass_2019 = df_tass_2019.rename(columns={'sentence': 'text', 'labels': 'label'})

# Unimos dfTwitterSentimentDataset con df_tass_2019
dfSentimentAnalysis = pd.concat([dfTwitterSentimentDataset, df_tass_2019], ignore_index=True)
# Mezclamos aleatoriamente los registros
dfSentimentAnalysis = dfSentimentAnalysis.sample(frac=1, random_state=12).reset_index(drop=True)

# Guardamos el dataset resultante para detección de sentimiento polarizado
dfSentimentAnalysis.to_csv('SentimentAnalysis_1.csv', index=False, encoding='utf-8')

# Imprimimos las dimensiones del dataset resultante y los últimos 10 ejemplos que posee
print(dfSentimentAnalysis.shape)
print(dfSentimentAnalysis.tail(10))

# Sección nueva

In [ ]:
# Dataset 3: Spanish Hate Speech Superset (Hugging Face)
# Link de referencia: https://huggingface.co/datasets/manueltonneau/spanish-hate-speech-superset
# Columns:
#   text: the annotated post
#   labels: annotation of whether the post is hateful (== 1) or not (==0). As datasets have different annotation schemes, we systematically binarized the labels.
#   source: origin of the data (e.g., Twitter)
#   dataset: dataset the data is from (see "Datasets" part below)
#   nb_annotators: number of annotators by post
#   tweet_id: tweet ID where available
#   post_author_country_location: post author country location, when it could be inferred.


df_SpHateSpeech = convert_dataset_to_dataframe("manueltonneau/spanish-hate-speech-superset")
df_SpHateSpeech.to_csv('SpHateSpeech.csv', index=False, encoding='utf-8')

In [ ]:
# Dataset 4: Contextualized Hate Speech (CONICET)
# Link de referencia:  https://datosdeinvestigacion.conicet.gov.ar/handle/11336/235509
# Columns:
#   id: id del tweet
#   title: titulo del tweet
#   text: contenido del tweet
#   context_tweet: contexto del tweet
#   HATEFUL: 1 si el tweet contiene discurso de odio, 0 si no
#   body:
#   CALLS: 1 si el tweet llama a la acción violenta, 0 si no
#   WOMEN: 1 si el tweet va en contra de las mujeres, 0 si no
#   LGBTI: 1 si el tweet va en contra de las personas LGBTI, 0 si no
#   RACISM: 1 si el tweet tiene un mensaje racista, 0 si no
#   CLASS: 1 si el tweet tiene un mensaje clasista, 0 si no
#   POLITICS: 1 si el tweet va en contra de la ideología política, 0 si no
#   DISABLED: 1 si el tweet va en contra de las personas con discapacidad, 0 si no
#   APPEARANCE: 1 si el tweet se mete con la apariencia de las personas, 0 si no
#   CRIMINAL: 1 si el tweet va en contra de los delincuentes o personas en conflicto con la ley, 0 si no

# Código utilizado para convertir los .parquet en el dataframe que necesitamos

# Importamos los datasets en formato .parquet
# df1 = pd.read_parquet('train-00000-of-00001.parquet')
# df2 = pd.read_parquet('dev-00000-of-00001.parquet')
# df3 = pd.read_parquet('test-00000-of-00001.parquet')

# Fusionamos los 3 datasets ingresados para obtener uno general y así preprocesar todos los registros en conjunto
# df = pd.concat([df1, df2, df3], ignore_index=True)

#print(df.columns)

# Eliminamos columnas id, title, context_tweet, y body dado que no son relevantes para nuestro futuro análisis
#df = df.drop(columns=['id', 'title', 'context_tweet', 'body'])

# Convertimos los datasets a formato .csv
# df.to_csv('CONICET_contextualized_hate_speech_DATA_reduced.csv', index=False, encoding='utf-8')

# A partir del dataset convertido inicialmente, generamos dos nuevos que contienen la información concreta que estaremos utilizando
df_CONICET_contextualized_hate_speech_DATA_reduced = pd.read_csv('CONICET_contextualized_hate_speech_DATA_reduced.csv')
print(df_CONICET_contextualized_hate_speech_DATA_reduced.columns)

# Primer dataset: dejamos las columnas text, HATEFUL que indica si el tweet expresa odio o no lo hace y CALLS que indica si el tweet hace un llamado a la acción violenta
# ---> Podemos entrenar un nuevo modelo para llamado a la acción armada, y luego en el "escenario real" crear la columna  CALL en el dataset 3 para ver como lo predice
df_CONICET_contextualized_hate_speech_DATA_reduced = df_CONICET_contextualized_hate_speech_DATA_reduced.drop(columns=['CLASS', 'WOMEN', 'LGBTI', 'RACISM','POLITICS', 'DISABLED', 'APPEARANCE', 'CRIMINAL'])
# df = df.rename(columns={'HATEFUL': 'hs'})
print('Columnas del dataset de discurso de odio + llamado a la acción violenta: ', df_CONICET_contextualized_hate_speech_DATA_reduced.columns)
df_CONICET_contextualized_hate_speech_DATA_reduced.to_csv('CONICET_contextualized_hate_speech_DATA_HSyCALLS.csv', index=False, encoding='utf-8')

# Segundo dataset: dejamos las columnas text y  HATEFUL que indica si el tweet expresa odio o no lo hace
df_CONICET_contextualized_hate_speech_DATA_HS = df_CONICET_contextualized_hate_speech_DATA_reduced.drop(columns=['CALLS'])
df_CONICET_contextualized_hate_speech_DATA_HS = df_CONICET_contextualized_hate_speech_DATA_HS.rename(columns={'HATEFUL': 'hs'})
print('Columnas del dataset de discurso de odio:', df_CONICET_contextualized_hate_speech_DATA_HS.columns)
df_CONICET_contextualized_hate_speech_DATA_HS.to_csv('CONICET_contextualized_hate_speech_DATA_HS.csv', index=False, encoding='utf-8')

# Imprimimos las dimensiones del dataset resultante y los últimos 10 ejemplos que posee
print(df_CONICET_contextualized_hate_speech_DATA_HS.shape)
print(df_CONICET_contextualized_hate_speech_DATA_HS.tail(10))

In [ ]:
# Dataset 4: Contextualized Hate Speech (CONICET)
# Link de referencia:  https://datosdeinvestigacion.conicet.gov.ar/handle/11336/235509
# Columns:
#   id: id del tweet
#   title: titulo del tweet
#   text: contenido del tweet
#   context_tweet: contexto del tweet
#   HATEFUL: 1 si el tweet contiene discurso de odio, 0 si no
#   body:
#   CALLS: 1 si el tweet llama a la acción violenta, 0 si no
#   WOMEN: 1 si el tweet va en contra de las mujeres, 0 si no
#   LGBTI: 1 si el tweet va en contra de las personas LGBTI, 0 si no
#   RACISM: 1 si el tweet tiene un mensaje racista, 0 si no
#   CLASS: 1 si el tweet tiene un mensaje clasista, 0 si no
#   POLITICS: 1 si el tweet va en contra de la ideología política, 0 si no
#   DISABLED: 1 si el tweet va en contra de las personas con discapacidad, 0 si no
#   APPEARANCE: 1 si el tweet se mete con la apariencia de las personas, 0 si no
#   CRIMINAL: 1 si el tweet va en contra de los delincuentes o personas en conflicto con la ley, 0 si no

# Código utilizado para convertir los .parquet en el dataframe que necesitamos

# Importamos los datasets en formato .parquet
# df1 = pd.read_parquet('train-00000-of-00001.parquet')
# df2 = pd.read_parquet('dev-00000-of-00001.parquet')
# df3 = pd.read_parquet('test-00000-of-00001.parquet')

# Fusionamos los 3 datasets ingresados para obtener uno general y así preprocesar todos los registros en conjunto
# df = pd.concat([df1, df2, df3], ignore_index=True)

#print(df.columns)

# Eliminamos columnas id, title, context_tweet, y body dado que no son relevantes para nuestro futuro análisis
#df = df.drop(columns=['id', 'title', 'context_tweet', 'body'])

# Convertimos los datasets a formato .csv
# df.to_csv('CONICET_contextualized_hate_speech_DATA_reduced.csv', index=False, encoding='utf-8')


# A partir del dataset convertido inicialmente, generamos dos nuevos que contienen la información concreta que estaremos utilizando
df_CONICET_contextualized_hate_speech_DATA_reduced = pd.read_csv(r"CONICET_contextualized_hate_speech_DATA_reduced.csv")
print(df_CONICET_contextualized_hate_speech_DATA_reduced.columns)

# Primer dataset: dejamos las columnas text, HATEFUL que indica si el tweet expresa odio o no lo hace y CALLS que indica si el tweet hace un llamado a la acción violenta
# ---> Podemos entrenar un nuevo modelo para llamado a la acción armada, y luego en el "escenario real" crear la columna  CALL en el dataset 3 para ver como lo predice
df_CONICET_contextualized_hate_speech_DATA_reduced = df_CONICET_contextualized_hate_speech_DATA_reduced.drop(columns=['CLASS', 'WOMEN', 'LGBTI', 'RACISM','POLITICS', 'DISABLED', 'APPEARANCE', 'CRIMINAL'])
# df = df.rename(columns={'HATEFUL': 'hs'})
print('Columnas del dataset de discurso de odio + llamado a la acción violenta: ', df_CONICET_contextualized_hate_speech_DATA_reduced.columns)
df_CONICET_contextualized_hate_speech_DATA_reduced.to_csv(r"CONICET_contextualized_hate_speech_DATA_HSyCALLS (2).csv", index=False, encoding='utf-8')
# Segundo dataset: dejamos las columnas text y  HATEFUL que indica si el tweet expresa odio o no lo hace
df_CONICET_contextualized_hate_speech_DATA_HS = df_CONICET_contextualized_hate_speech_DATA_reduced.drop(columns=['CALLS'])
df_CONICET_contextualized_hate_speech_DATA_HS = df_CONICET_contextualized_hate_speech_DATA_HS.rename(columns={'HATEFUL': 'hs'})
print('Columnas del dataset de discurso de odio:', df_CONICET_contextualized_hate_speech_DATA_HS.columns)
df_CONICET_contextualized_hate_speech_DATA_HS.to_csv(r"CONICET_contextualized_hate_speech_DATA_HS (1).csv", index=False, encoding='utf-8')

# Imprimimos las dimensiones del dataset resultante y los últimos 10 ejemplos que posee
print(df_CONICET_contextualized_hate_speech_DATA_HS.shape)
print(df_CONICET_contextualized_hate_speech_DATA_HS.tail(10))

In [ ]:
# Creación del dataset para detección de contenido de odio
# Fusionamos df_SpHateSpeech con df_CONICET_contextualized_hate_speech_DATA_HS

# Revisamos las columnas de df_SpHateSpeech
print(df_SpHateSpeech.columns)

# Eliminamos las columnas source, dataset, nb_annotators, tweet_id, post_author_country_location dado que no vamos a necesitarla
#df_SpHateSpeech = df_SpHateSpeech.drop(columns=['source', 'dataset', 'nb_annotators', 'tweet_id','post_author_country_location'])

# Renombramos las columnas del segundo dataset para que coincidan con las del primero
df_SpHateSpeech = df_SpHateSpeech.rename(columns={'labels': 'hs'})
df_SpHateSpeech['hs'] = df_SpHateSpeech['hs'].astype(int)

# Unimos df_SpHateSpeech con df_CONICET_contextualized_hate_speech_DATA_HS
dfHateSpeech = pd.concat([df_SpHateSpeech, df_CONICET_contextualized_hate_speech_DATA_HS], ignore_index=True)
# Mezclamos aleatoriamente los registros
dfHateSpeech = dfHateSpeech.sample(frac=1, random_state=18).reset_index(drop=True)

# Guardamos el dataset resultante para detección de sentimiento polarizado
dfHateSpeech.to_csv('HateSpeech_1.csv', index=False, encoding='utf-8')

# Imprimimos las dimensiones del dataset resultante y los últimos 10 ejemplos que posee
print(dfHateSpeech.shape)
print(dfHateSpeech.tail(10))

In [ ]:
# Creación del dataset para detección de contenido de odio
# Fusionamos df_SpHateSpeech con df_CONICET_contextualized_hate_speech_DATA_HS

# Revisamos las columnas de df_SpHateSpeech
print(df_SpHateSpeech.columns)

# Eliminamos las columnas source, dataset, nb_annotators, tweet_id, post_author_country_location dado que no vamos a necesitarla
#df_SpHateSpeech = df_SpHateSpeech.drop(columns=['source', 'dataset', 'nb_annotators', 'tweet_id','post_author_country_location'])

# Renombramos las columnas del segundo dataset para que coincidan con las del primero
df_SpHateSpeech = df_SpHateSpeech.rename(columns={'labels': 'hs'})
df_SpHateSpeech['hs'] = df_SpHateSpeech['hs'].astype(int)

# Unimos df_SpHateSpeech con df_CONICET_contextualized_hate_speech_DATA_HS
dfHateSpeech = pd.concat([df_SpHateSpeech, df_CONICET_contextualized_hate_speech_DATA_HS], ignore_index=True)
# Mezclamos aleatoriamente los registros
dfHateSpeech = dfHateSpeech.sample(frac=1, random_state=18).reset_index(drop=True)

# Guardamos el dataset resultante para detección de sentimiento polarizado
dfHateSpeech.to_csv('HateSpeech_1.csv', index=False, encoding='utf-8')

# Imprimimos las dimensiones del dataset resultante y los últimos 10 ejemplos que posee
print(dfHateSpeech.shape)
print(dfHateSpeech.tail(10))

In [ ]:
# Modificaciones previas en los datasets conformados para los análisis

# Cargamos los corpus con los que estaremos trabajando
dfSentimentAnalysis = pd.read_csv('SentimentAnalysis_1.csv')
dfHateSpeech = pd.read_csv('HateSpeech_1.csv')

# Eliminamos en el corpus de sentimiento los registros correspondientes a la categoría 2 (aquellos que no pudieron clasificarse correctamente)
dfSentimentAnalysis = dfSentimentAnalysis[dfSentimentAnalysis['label'] != 2]

# Renombramos la columna 'label' por 'labels' para mantener concordancia con los recursos que utilizamores
# labels: 1 positive, 0 negative, 2 neutral
dfSentimentAnalysis = dfSentimentAnalysis.rename(columns={"label": "labels"})
dfHateSpeech = dfHateSpeech.rename(columns={"hs": "labels"})

# En el dataset de análisis de sentimiento reemplazar la etiqueta 3 por 2
dfSentimentAnalysis['labels'] = dfSentimentAnalysis.apply(
    lambda x: 2 if x['labels'] == 3 else x['labels'], axis=1
)

# Voy ahora a conformar dos datasets de 72.000 tweets cada uno a partir de los importados
# Estos datasets tendrán mejor proporcionadas las distribuciones de los tweets de acuerdo a sus clases

# Dataframe de sentimiento
negativos = dfSentimentAnalysis[dfSentimentAnalysis['labels'] == 0].sample(n=24000, random_state=42)
neutrales  = dfSentimentAnalysis[dfSentimentAnalysis['labels'] == 1].sample(n=24000, random_state=42)
positivos  = dfSentimentAnalysis[dfSentimentAnalysis['labels'] == 2].sample(n=24000, random_state=42)
dfSentimentAnalysis_balanced = pd.concat([negativos, neutrales, positivos])
# Mezclamos aleatoriamente el orden de los tweets en el dataset balanceado
dfSentimentAnalysis_balanced = dfSentimentAnalysis_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

# Dataframe de discurso de odio
no_odio = dfHateSpeech[dfHateSpeech['labels'] == 0].sample(n=56020, random_state=42)
odio = dfHateSpeech[dfHateSpeech['labels'] == 1].sample(n=15980, random_state=42)
dfHateSpeech_balanced = pd.concat([no_odio, odio])
# Mezclamos aleatoriamente el orden de los tweets en el dataset balanceado
dfHateSpeech_balanced = dfHateSpeech_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

# Revisamos las proporciones de los nuevos datasets conformados
print(dfSentimentAnalysis_balanced['labels'].value_counts())
print(dfHateSpeech_balanced['labels'].value_counts())

# Guardamos las nuevas versiones de los datasets  dataset resultante para detección de sentimiento polarizado
dfSentimentAnalysis_balanced.to_csv('SentimentAnalysis.csv', index=False, encoding='utf-8')
dfHateSpeech_balanced.to_csv('HateSpeech.csv', index=False, encoding='utf-8')

# Preprocesamiento de los corpus
En este punto comienza el preprocesamiento de los corpus conformados en el apartado anterior, así como un análisis exploratorio inicial de los textos que los conforman.

In [ ]:
# Preprocesamiento de ambos datasets

# Cargamos los corpus con los que estaremos trabajando
dfSentimentAnalysis = pd.read_csv('SentimentAnalysis.csv')
sentiment_map = { 0: 'negativo', 1: 'neutral', 2: 'positivo' }
dfSentimentAnalysis['labels'] = dfSentimentAnalysis['labels'].map(sentiment_map)

dfHateSpeech = pd.read_csv('HateSpeech.csv')
hate_map = { 0: 'no hate', 1: 'hate' }
dfHateSpeech['labels'] = dfHateSpeech['labels'].map(hate_map)

# Analisis Exploratorio inicial
print('Dimensiones del corpus de Análisis de Sentimiento: ', dfSentimentAnalysis.shape)

print('Cantidades de ejemplos por cada clase de sentimiento:')
print(dfSentimentAnalysis['labels'].value_counts())

print('\nDimensiones del corpus de Detección de Contenido de Odio: ', dfHateSpeech.shape)
print('Cantidades de ejemplos que contienen (y que no contienen) discurso de odio:')
print(dfHateSpeech['labels'].value_counts())

# Guardamos los índices originales para trazabilidad
indexes_sent_bow = dfSentimentAnalysis.index
indexes_hate_bow = dfHateSpeech.index

In [ ]:
# Exploramos la dispersión de los corpus con respecto a las clases de los tweets que lo comprenden
print(dfSentimentAnalysis['labels'].value_counts(normalize=True))
print(dfHateSpeech['labels'].value_counts(normalize=True))

In [ ]:
# Comenzamos el preprocesamiento para baseline

# Normalizamos el texto:

# Pasamos todos los tweets a minusculas
# Eliminamos caracteres especiales y emojis
# Eliminamos URLs y menciones a usuarios y hashtags

def normalization(df):
  # Pasamos todos los tweets a minusculas
  df['text'] = df['text'].str.lower()
  # Eliminamos caracteres especiales y emojis
  # Eliminamos emojis
  df['text'] = df['text'].str.replace(emoji_pattern, '', regex=True)
  # Eliminamos URLs
  df['text'] = df['text'].str.replace(r'http\S+|www.\S+|\b\w+\.(com|net|org)\b', '', regex=True)
  # Eliminamos menciones a usuarios
  df['text'] = df['text'].str.replace(r'@\w+', '', regex=True)
  # Eliminamos hashtags
  df['text'] = df['text'].str.replace(r'#\w+', '', regex=True)

  # Los tweets están en español: mantenemos tildes y algunas letras especiales y signos de puntuación
  df['text'] = df['text'].str.replace(r'[^a-zA-Z0-9áéíóúüñÁÉÍÓÚÜÑ¿?¡!.,\s]', '', regex=True)
  # Eliminamos espacios en blanco del comienzo y del final de cada texto
  df['text'] = df['text'].str.strip()
  return df

# Normalizamos los textos de los corpus
dfSentimentAnalysis_normalized = normalization(dfSentimentAnalysis)
dfHateSpeech_normalized = normalization(dfHateSpeech)

# Imprimimos los últimos 10 tweets normalizados de cada corpus
print(dfSentimentAnalysis_normalized.tail(10), '\n\n')
print(dfHateSpeech_normalized.tail(10), '\n\n')

In [ ]:
# Eliminación de stopwords
stop_words = set(stopwords.words('spanish'))

def remove_stopwords(df):
  # Convertimos todos los valores a cadenas de texto
  df['text'] = df['text'].fillna('').astype(str)
  # Eliminamos las stopwords
  df['text'] = df['text'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stop_words) and len(word) > 2]))
  return df

# Sacamos los stopwords de los textos de los corpus
dfSentimentAnalysis_without_stopwords = remove_stopwords(dfSentimentAnalysis_normalized)
dfHateSpeech_without_stopwords = remove_stopwords(dfHateSpeech_normalized)

# Imprimimos los últimos 10 tweets normalizados de cada corpus
print(dfSentimentAnalysis_without_stopwords.tail(10), '\n\n')
print(dfHateSpeech_without_stopwords.tail(10), '\n\n')

Esto ya entra para el preprocesamiento específico del baseline de nuestro análisis

In [ ]:
# Para reducir las palabras de los corpus y así mejorar el preprocesamiento:
# O se usa Stemming o se usa Lematización

def stemming(df):
    # Preparar el Stemmer de NLTK
    # stemmer = SnowballStemmer("spanish")
    stemmer = nltk.stem.SnowballStemmer('spanish')
    # Tokeniza mediante Stemming
    df['text'] = df['text'].apply(lambda text:
        ' '.join([stemmer.stem(token) for token in nltk.word_tokenize(text, language='spanish')])
    )
    return df

def lemmatization(df):
    # Cargar el modelo spaCy para lematización
    nlp = spacy.load("es_core_news_sm")
    # Tokeniza mediante Lematizacion
    with nlp.disable_pipes("parser", "ner"):
        df['text'] = df['text'].apply(lambda text:
            ' '.join([token.lemma_ for token in nlp(text)])
        )
    return df

def custom_tokenizer(df, lemmatize=True):
    return lemmatization(df) if lemmatize else stemming(df)

dfSentimentAnalysis_token = custom_tokenizer(dfSentimentAnalysis_without_stopwords)
dfHateSpeech_token = custom_tokenizer(dfHateSpeech_without_stopwords)

In [ ]:
# Para los modeos clásicos de ML que utilizaremos como baseline (Naive Bayes, Logistic Regression, SVM)
# debemos vectorizar el texto, ya que estos modelos no pueden manejar texto directamente, por lo que
# necesitan una representación numérica

# TF-IDF calcula el peso de cada palabra en cada documento según su frecuencia relativa en ese documento y su frecuencia inversa en todo el corpus
def tfidf_vectorizer(corpus):
    # params max_features=5000, ngram_range=(1,2) para que tenga en cuenta las 5000 palabras más importantes y ngramas de 1 y 2 palabras
    vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,3)) # tokenizer=lambda text: custom_tokenizer(text, lemmatize=True) como parámetro para que haga lematizacion/stemming todo junto
    X = vectorizer.fit_transform(corpus)
    return X, vectorizer

# Vectorización implicita mediante la vectorización de los n-gramas de los corpus con TF-IDF
X_sent_tfidf, vect_sent_tfidf = tfidf_vectorizer(dfSentimentAnalysis_token['text'])
X_hate_tfidf, vect_hate_tfidf = tfidf_vectorizer(dfHateSpeech_token['text'])

# Vemos el vocabulario generado
print(vect_sent_tfidf.get_feature_names_out(), ' \ncantidad de elementos de dfSent con tfidf: ', len(vect_sent_tfidf.get_feature_names_out()), '\n')
print(vect_hate_tfidf.get_feature_names_out(), ' \ncantidad de elementos de dfHate con tfidf: ', len(vect_hate_tfidf.get_feature_names_out()), '\n')

In [ ]:
# Creamos dos series con las etiquetas de los labels: uno para cada corpus
dfSentimentAnalysis_y = dfSentimentAnalysis_token['labels']
dfHateSpeech_y = dfHateSpeech_token['labels']

# Comienza la sección de evaluacion de modelos de baseline

In [ ]:
# Separación de los datos en conjuntos de entrenamiento y testeo
# Como las clases están dispersas aplicamos el parámetro stratify para mantener esta dispersión pareja en los conjuntos resultantes

def split_data(X, y, indexes, test_size=0.1, random_state=42):
    return train_test_split(X, y, indexes, test_size=test_size, stratify=y, random_state=random_state)

In [ ]:
# Entrenamos y validamos los modelos utilizando Cross-Validation K-fold para clases desbalanceadas (StratifiedKFold)
# Particionamos el conjunto en 10 particiones

# Obs: average=macro calcula la métrica individualmente para cada clase y luego promedia los valores, dando igual peso a todas las clases, sin importar cuántas muestras haya de cada una

def cross_validate_model(model, X, y, cv=10):
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    metrics = {'accuracy': [], 'f1_macro': [], 'precision_macro': [], 'recall_macro': []}

    # En cada iteracion entrena utilizando k-1 folds y valida con el k restante, así va ajustando el modelo final
    for train_index, val_index in skf.split(X, y):
        X_train_cv, X_val_cv = X[train_index], X[val_index]
        y_train_cv, y_val_cv = y[train_index], y[val_index]

        model.fit(X_train_cv, y_train_cv)
        y_pred = model.predict(X_val_cv)

        # Vamos guardando las métricas resultantes de las k-esimas predicciones
        metrics['accuracy'].append(accuracy_score(y_val_cv, y_pred))
        metrics['f1_macro'].append(f1_score(y_val_cv, y_pred, average='macro'))
        metrics['precision_macro'].append(precision_score(y_val_cv, y_pred, average='macro'))
        metrics['recall_macro'].append(recall_score(y_val_cv, y_pred, average='macro'))

    # Retornamos un diccionario que tiene, por cada métrica, el promedio y el desvío estandar que tuvieron en las k iteraciones
    return {m: (np.mean(metrics[m]), np.std(metrics[m])) for m in metrics}

In [ ]:
# Matriz de confusión para cada evaluación final
def plot_confusion_matrix(y_true, y_pred, labels=None, title="Matriz de Confusión"):
    cm = confusion_matrix(y_true, y_pred, labels=labels) if labels is not None else confusion_matrix(y_true, y_pred)


    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.show()

# Hora de la verdad: evaluación final de los modelos
def evaluate_model(model, X_train, y_train, X_test, y_test, label_list):

    # Evaluación del modelo y realización de las predicciones
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Métricas de la evaluación del modelo
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'f1_macro': f1_score(y_test, y_pred, average='macro'),
        'precision_macro': precision_score(y_test, y_pred, average='macro'),
        'recall_macro': recall_score(y_test, y_pred, average='macro'),
    }

    # Generamos un resumen de las métricas de las predicciones por clase en formato de tabla
    print(classification_report(y_test, y_pred, labels=label_list, target_names=label_list, digits=2))
    # Mapa de calor para matriz de confusion
    plot_confusion_matrix(y_test, y_pred, labels=label_list, title=f"Matriz de Confusión — {model.__class__.__name__}")

    return metrics, y_pred

In [ ]:
# Para TfIdf de corpus de sentimiento
(X_train_sent_tfidf, X_test_sent_tfidf, y_train_sent_tfidf, y_test_sent_tfidf, idx_train_sent_tfidf, idx_test_sent_tfidf) = split_data(X_sent_tfidf, dfSentimentAnalysis_y, indexes_sent_bow)
# Para TfIdf de corpus de discurso de odio
(X_train_hate_tfidf, X_test_hate_tfidf, y_train_hate_tfidf, y_test_hate_tfidf, idx_train_hate_tfidf, idx_test_hate_tfidf) = split_data(X_hate_tfidf, dfHateSpeech_y, indexes_hate_bow)


# Evaluamos que se hayan mantenido las proporciones entre las clases de cada corpus
print(y_train_sent_tfidf.value_counts(normalize=True))
print(y_test_sent_tfidf.value_counts(normalize=True), '\n')

print(y_train_hate_tfidf.value_counts(normalize=True))
print(y_test_hate_tfidf.value_counts(normalize=True), '\n')

In [ ]:
# Agregamos resultados al dataframe
def agregar_resultados(df, corpus_name, model_name, conjunto, metrics):
    df.loc[len(df)] = {
        'Corpus': corpus_name,
        'Modelo': model_name,
        'Conjunto': conjunto,
        'Accuracy': metrics['accuracy'],
        'F1 Macro': metrics['f1_macro'],
        'Precision Macro': metrics['precision_macro'],
        'Recall Macro': metrics['recall_macro']
    }
    return df

In [ ]:
# Definimos los modelos que utilizaremos para el baseline
# MultinomialNB es el clasificador más adecuado para las vectorizaciones tfidf

models = {
    'Naive Bayes': MultinomialNB(), # random_state=17
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=17),
    'SVM': LinearSVC(random_state=17)
}

corpus_sets = {
    'Sentiment TF-IDF': (X_train_sent_tfidf, X_test_sent_tfidf, y_train_sent_tfidf, y_test_sent_tfidf, idx_train_sent_tfidf, idx_test_sent_tfidf),
    'Hate TF-IDF': (X_train_hate_tfidf, X_test_hate_tfidf, y_train_hate_tfidf, y_test_hate_tfidf, idx_train_hate_tfidf, idx_test_hate_tfidf)
}

# Dataframe resumen que almacenará los resultados de las métricas
resultados_baseline = pd.DataFrame(columns=[
    'Corpus', 'Modelo', 'Conjunto', 'Accuracy', 'F1 Macro', 'Precision Macro', 'Recall Macro'
])

# Definimos dataframe de trazabilidad para los tweets de evaluación de los corpus
traceability_df = pd.DataFrame(columns=['Corpus', 'Tweet_Index', 'Modelo', 'Label', 'Predicted'])

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Cross-validation y evaluacion final para cada corpus definido, en cada uno de los modelos definidos

for corpus_name, (X_train, X_test, y_train, y_test, idx_train, idx_test) in corpus_sets.items():
    print(f"\n{'-'*60}\nCorpus: {corpus_name}\n{'-'*60}")

    if corpus_name.lower().startswith('hate'):
        etiquetas = ['no hate', 'hate']
    else:
        etiquetas = ['negativo', 'neutral', 'positivo']

    for model_name, model in models.items():
        print(f"\n--- Cross-validation para {model_name} ---")
        results_cv = cross_validate_model(model, X_train, y_train.values)
        print(results_cv)

        # Agregamos métricas de cross-validation al dataframe
        resultados_baseline = agregar_resultados(resultados_baseline, corpus_name, model_name, 'Cross-Validation', {
            'accuracy': results_cv['accuracy'][0],
            'f1_macro': results_cv['f1_macro'][0],
            'precision_macro': results_cv['precision_macro'][0],
            'recall_macro': results_cv['recall_macro'][0]
        })

        print(f"\n--- Evaluación final para {model_name} ---")
        resultados_evaluacion, y_pred = evaluate_model(model, X_train, y_train, X_test, y_test, label_list= etiquetas)
        print(resultados_evaluacion)

        # Agregamos métricas de evaluación final al dataframe
        resultados_baseline = agregar_resultados(resultados_baseline, corpus_name, model_name, 'Test Final', resultados_evaluacion)

        # Guardamos trazabilidad por tweet
        temp_trace_df = pd.DataFrame({
            'Corpus': corpus_name,
            'Tweet_Index': idx_test,  # los índices originales
            'Modelo': model_name,
            'Label': y_test,
            'Predicted': y_pred
        })

        traceability_df = pd.concat([traceability_df, temp_trace_df], ignore_index=True)

In [ ]:
# Visualizaciones de las métricas del baseline

def plot_resultados_baseline(df_resultados):
    metrics = ['Accuracy', 'F1 Macro', 'Precision Macro', 'Recall Macro']
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.flatten()

    for i, metric in enumerate(metrics):
        sns.barplot(data=df_resultados, x='Modelo', y=metric, hue='Conjunto', ax=axes[i])
        axes[i].set_title(f'{metric} por Modelo')
        axes[i].set_ylim(0.5, 1)
        axes[i].legend(loc='lower right')
        axes[i].set_xlabel('')
        axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=15)

    plt.suptitle('Comparación de métricas de los modelos baseline', fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

# Mostramos resumen completo
display(resultados_baseline)

# Mostramos gráficos comparativos
plot_resultados_baseline(resultados_baseline)

In [ ]:
# Visualizamos la trazabilidad
traceability_df.head()

# Exportamos la trazabilidad
# traceability_df.to_csv("baseline_traceability.csv", index=False)

In [ ]:
from sklearn.metrics import classification_report

def reporte_clasificacion_por_modelo(corpus_name,
                                     X_train, X_test,
                                     y_train, y_test,
                                     model_name, model):
    # 1) Entrenamos y predecimos
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # 2) Sacamos las clases que hay en este y_test
    clases_presentes = sorted(y_test.unique())

    # 3) Si son numéricas, las mapeamos a nombres; si ya son strings, las usamos tal cual
    #    Y en todo caso construimos target_names en paralelo.
    if np.issubdtype(y_test.dtype, np.number):
        if corpus_name.lower().startswith('hate'):
            nombre_clase = {0: "no hate", 1: "hate"}
        else:
            nombre_clase = {0: "negativo", 1: "neutral", 2: "positivo"}
        labels = clases_presentes
        target_names = [nombre_clase[c] for c in clases_presentes]
    else:
        # y_test ya trae strings: 'negativo','neutral','positivo' o 'no hate','hate'
        labels = clases_presentes
        target_names = clases_presentes

    # 4) Mostramos el reporte
    print(f"Reporte de clasificación para {model_name} en corpus {corpus_name}:\n")
    print(classification_report(
        y_test, y_pred,
        labels=labels,
        target_names=target_names,
        digits=2
    ))


In [ ]:
# Repetido para más claridad al analizar las métricas por separado

for corpus_name, (X_train, X_test, y_train, y_test, *_ ) in corpus_sets.items():
    for model_name, model in models.items():
        reporte_clasificacion_por_modelo(
            corpus_name,
            X_train, X_test,
            y_train, y_test,
            model_name, model
        )

In [ ]:
X_train_sent_tfidf, X_test_sent_tfidf, y_train_sent_tfidf, y_test_sent_tfidf, _, _ = corpus_sets['Sentiment TF-IDF']
from sklearn.linear_model import LogisticRegression

# Logistic Regression para soporte multiclase (3 clases)
models['Logistic Regression'] = LogisticRegression(
    solver='lbfgs',
    max_iter=1000
)

# Entrenamos el modelo con el conjunto TF-IDF
models['Logistic Regression'].fit(X_train_sent_tfidf, y_train_sent_tfidf)

from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

def plot_roc_auc(model, X_test, y_test, classes):
    # Mapa de colores
    color_map = {'negativo':'red', 'neutral':'blue', 'positivo':'green'}
    # Binarizamos los labels
    y_test_bin = label_binarize(y_test, classes=classes)
    # Probabilidades de cada clase
    y_score = model.predict_proba(X_test)

    plt.figure(figsize=(10, 8))
    for i, cls in enumerate(classes):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        # usamos cls directamente
        plt.plot(
            fpr, tpr,
            color=color_map[cls],   # cls es 'negativo', 'neutral' o 'positivo'
            label=f"{cls} (AUC = {roc_auc:.2f})",
            linewidth=2
        )
    plt.plot([0, 1], [0, 1], 'k--', linewidth=1)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Curvas ROC por clase - Logistic Regression (TF-IDF)')
    plt.legend(loc='lower right')
    plt.grid()
    plt.show()

classes = sorted(y_train_sent_tfidf.unique())  # ['negativo','neutral','positivo']
plot_roc_auc(models['Logistic Regression'], X_test_sent_tfidf, y_test_sent_tfidf, classes)


In [ ]:
from sklearn.model_selection import learning_curve
import numpy as np

train_sizes, train_scores, test_scores = learning_curve(
    models['Logistic Regression'], X_train_sent_tfidf, y_train_sent_tfidf, cv=5,
    scoring='accuracy', n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10)
)

plt.figure(figsize=(10, 8))
plt.plot(train_sizes, np.mean(train_scores, axis=1), 'o-', color='r', label='Train score')
plt.plot(train_sizes, np.mean(test_scores, axis=1), 'o-', color='g', label='Validation score')
plt.title('Curvas de aprendizaje')
plt.xlabel('Tamaño de entrenamiento')
plt.ylabel('Exactitud')
plt.legend()
plt.grid()
plt.show()


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt

models = {}

# Entrenar modelo con TF-IDF
models['Logistic Regression TFIDF'] = LogisticRegression(solver='lbfgs', max_iter=1000)
models['Logistic Regression TFIDF'].fit(X_train_sent_tfidf, y_train_sent_tfidf)

def compute_metrics(y_true, y_pred):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'Recall': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'F1-score': f1_score(y_true, y_pred, average='weighted', zero_division=0)
    }

def plot_metrics_bar(y_true, y_pred, title):
    metrics = compute_metrics(y_true, y_pred)
    names = list(metrics.keys())
    values = list(metrics.values())

    plt.figure(figsize=(8,5))
    bars = plt.bar(names, values, color=['skyblue', 'orange', 'green', 'red'])
    plt.ylim(0, 1)
    plt.title(title)
    plt.ylabel('Score')
    plt.grid(axis='y')

    # Mostrar valor encima de cada barra
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval + 0.02, f'{yval:.2f}', ha='center')

    plt.show()

# Evaluar y graficar para TF-IDF
y_pred_tfidf = models['Logistic Regression TFIDF'].predict(X_test_sent_tfidf)
plot_metrics_bar(y_test_sent_tfidf, y_pred_tfidf, 'Métricas para Logistic Regression con TF-IDF')


### Conformación de los datasets que utilizaremos en las etapas de fine tuning y de context learning

In [ ]:
# Datasets base para fine tuning: 60.000 muestras por cada dataset
dfSentimentAnalysis_train = dfSentimentAnalysis_without_stopwords.loc[idx_train_sent_tfidf]
dfSentimentAnalysis_test = dfSentimentAnalysis_without_stopwords.loc[idx_test_sent_tfidf]
dfHateSpeech_train = dfHateSpeech_without_stopwords.loc[idx_train_hate_tfidf]
dfHateSpeech_test = dfHateSpeech_without_stopwords.loc[idx_test_hate_tfidf]

# Dataframe de sentimiento: entrenamiento y evaluacion para fine tuning
negativos_ft_train = dfSentimentAnalysis_train[dfSentimentAnalysis_train['labels'] == 0].sample(n=18000, random_state=42)
neutrales_ft_train = dfSentimentAnalysis_train[dfSentimentAnalysis_train['labels'] == 1].sample(n=18000, random_state=42)
positivos_ft_train = dfSentimentAnalysis_train[dfSentimentAnalysis_train['labels'] == 2].sample(n=18000, random_state=42)
dfSentimentAnalysis_ft_train = pd.concat([negativos_ft_train, neutrales_ft_train, positivos_ft_train])
dfSentimentAnalysis_ft_train = dfSentimentAnalysis_ft_train.sample(frac=1, random_state=42).reset_index(drop=True)

negativos_ft_test = dfSentimentAnalysis_test[dfSentimentAnalysis_test['labels'] == 0].sample(n=2000, random_state=42)
neutrales_ft_test = dfSentimentAnalysis_test[dfSentimentAnalysis_test['labels'] == 1].sample(n=2000, random_state=42)
positivos_ft_test = dfSentimentAnalysis_test[dfSentimentAnalysis_test['labels'] == 2].sample(n=2000, random_state=42)
dfSentimentAnalysis_ft_test = pd.concat([negativos_ft_test, neutrales_ft_test, positivos_ft_test])
dfSentimentAnalysis_ft_test = dfSentimentAnalysis_ft_test.sample(frac=1, random_state=42).reset_index(drop=True)

# Dataframe de discurso de odio: entrenamiento y evaluacion para fine tuning
no_odio_ft_train = dfHateSpeech_train[dfHateSpeech_train['labels'] == 0].sample(n=42015, random_state=42)
odio_ft_train = dfHateSpeech_train[dfHateSpeech_train['labels'] == 1].sample(n=11985, random_state=42)
dfHateSpeech_ft_train = pd.concat([no_odio_ft_train, odio_ft_train])
dfHateSpeech_ft_train = dfHateSpeech_ft_train.sample(frac=1, random_state=42).reset_index(drop=True)

no_odio_ft_test = dfHateSpeech_test[dfHateSpeech_test['labels'] == 0].sample(n=4668, random_state=42)
odio_ft_test = dfHateSpeech_test[dfHateSpeech_test['labels'] == 1].sample(n=1332, random_state=42)
dfHateSpeech_ft_test = pd.concat([no_odio_ft_test, odio_ft_test])
dfHateSpeech_ft_test = dfHateSpeech_ft_test.sample(frac=1, random_state=42).reset_index(drop=True)

# Guardamos las nuevas versiones de los datasets  resultante para detección de sentimiento polarizado en fine tuning
dfSentimentAnalysis_ft_train.to_csv('dfSentimentAnalysis_ft_train.csv', index=False, encoding='utf-8')
dfSentimentAnalysis_ft_test.to_csv('dfSentimentAnalysis_ft_test.csv', index=False, encoding='utf-8')
dfHateSpeech_ft_train.to_csv('dfHateSpeech_ft_train.csv', index=False, encoding='utf-8')
dfHateSpeech_ft_test.to_csv('dfHateSpeech_ft_test.csv', index=False, encoding='utf-8')

# Datasets base para context learning: 30 muestras por cada dataset
dfSentimentAnalysis_test_cl = dfSentimentAnalysis.loc[idx_test_sent_tfidf]
dfHateSpeech_test_cl = dfHateSpeech.loc[idx_test_hate_tfidf]

# Dataframe de sentimiento: evaluacion para context learning
negativos_cl_test = dfSentimentAnalysis_test[dfSentimentAnalysis_test['labels'] == 0].sample(n=10, random_state=42)
neutrales_cl_test = dfSentimentAnalysis_test[dfSentimentAnalysis_test['labels'] == 1].sample(n=10, random_state=42)
positivos_cl_test = dfSentimentAnalysis_test[dfSentimentAnalysis_test['labels'] == 2].sample(n=10, random_state=42)
dfSentimentAnalysis_cl_test = pd.concat([negativos_cl_test, neutrales_cl_test, positivos_cl_test])

# Dataframe de discurso de odio: evaluacion para context learning
no_odio_cl_test = dfHateSpeech_test[dfHateSpeech_test['labels'] == 0].sample(n=15, random_state=42)
odio_cl_test = dfHateSpeech_test[dfHateSpeech_test['labels'] == 1].sample(n=15, random_state=42)
dfHateSpeech_cl_test = pd.concat([no_odio_cl_test, odio_cl_test])

# Guardamos las nuevas versiones de los datasets  resultante para detección de sentimiento polarizado en context learning
dfSentimentAnalysis_cl_test.to_csv('dfSentimentAnalysis_cl_test.csv', index=False, encoding='utf-8')
dfHateSpeech_cl_test.to_csv('dfHateSpeech_cl_test.csv', index=False, encoding='utf-8')

## Pysentimiento y Fine Tuning

Referencias:

* Pysentimiento: https://github.com/pysentimiento/pysentimiento


In [ ]:
# Cargamos los datasets que utilizaremos en el fine tuning
df_train_sa = pd.read_csv('dfSentimentAnalysis_ft_train.csv')
df_test_sa = pd.read_csv('dfSentimentAnalysis_ft_test.csv')
df_train_hs = pd.read_csv('dfHateSpeech_ft_train.csv')
df_test_hs = pd.read_csv('dfHateSpeech_ft_test.csv')

## Analizador preentrenado: Sentiment Analysis

In [ ]:
# Creamos analizador preentrenado para análisis de sentimientos
sentiment_analyzer = create_analyzer(task="sentiment", lang="es")

In [ ]:
# Predecimos sobre el dataset de testeo de análisis de sentimientos
df_test_sa = df_test_sa.dropna(subset=["text"])
df_test_sa["predicted"] = df_test_sa["text"].apply(lambda x: sentiment_analyzer.predict(x).output)

In [ ]:
# Guardamos el dataset con las comparativas de rendimiento de pysentimiento frente a valores esperados
df_test_sa.to_csv('pysSentAnalysis.csv', index=False, encoding='utf-8')

In [ ]:
df_test_sa["predicted"] = df_test_sa["predicted"].replace({
    "NEG": 0,
    "NEU": 2,
    "POS": 1
})

In [ ]:
# Reporte de métricas para analisis de sentimientos con pysentimiento
print("Análisis de Sentimiento — Métricas")
print(classification_report(df_test_sa["labels"], df_test_sa["predicted"]))

In [ ]:
# Matriz de confusión para análisis de sentimientos con pysentimiento
print("Matriz de Confusión:")
print(confusion_matrix(df_test_sa["labels"], df_test_sa["predicted"]))

## Analizador preentrenado: Hate Speech

In [ ]:
# Creamos analizador preentrenado para detección de discurso de odio
sentiment_analyzer = create_analyzer(task="sentiment", lang="es")

In [ ]:
# Predecimos sobre el dataset de testeo de deteccion discurso de odio
df_test_hs = df_test_hs.dropna(subset=["text"])
df_test_hs["predicted"] = df_test_hs["text"].apply(lambda x: sentiment_analyzer.predict(x).output)

In [ ]:
# Guardamos el dataset con las comparativas de rendimiento de pysentimiento frente a valores esperados
df_test_hs.to_csv('pysHateSpeech.csv', index=False, encoding='utf-8')

In [ ]:
df_test_hs["predicted"] = df_test_hs["predicted"].replace({
    "NEG": 1,
    "NEU": 0,
    "POS": 0
})

In [ ]:
# Reporte de métricas para deteccion de discurso de odio con pysentimiento
print("Análisis de Sentimiento — Métricas")
print(classification_report(df_test_hs["labels"], df_test_hs["predicted"]))

In [ ]:
# Matriz de confusión para deteccion de discurso de odio con pysentimiento
print("Matriz de Confusión:")
print(confusion_matrix(df_test_hs["labels"], df_test_hs["predicted"]))

In [ ]:
# Descargar los archivos a tu PC:
# !zip -r /content/colab_backup.zip /content

# from google.colab import files
# files.download('/content/colab_backup.zip')

Antes de ver las métricas de abajo, revisar primero los notebooks donde realizamos nuestro fine tuning de Pysentimiento y donde hacemos el Context Learning con Gemini.

# Métricas Finales Obtenidas

In [ ]:
metricas = pd.read_csv('metricas_finales.csv')

metricas